# RNA — Aiuti di Stato alle imprese italiane

Dataset: `rna_aiuti_stato` (MIMIT — Registro Nazionale Aiuti di Stato)

Analisi pubblica: [README](../README.md)

Ogni aiuto pubblico concesso alle imprese italiane: beneficiario, importo, concedente, procedimento, settore NACE, regione. 17 milioni di righe, 2017-2026.

In [1]:
import duckdb
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-east-1';")
con.execute("SET s3_access_key_id='';")
con.execute("SET s3_secret_access_key='';")
con.execute("SET s3_session_token='';")

GCS_PATH = 'gs://dataciviclab-clean/rna_aiuti_stato/*/rna_aiuti_stato_*_clean.parquet'
print('DuckDB ready, GCS anonymous access configured')

DuckDB ready, GCS anonymous access configured


In [2]:
# 1. Trend annuale — importi e numero di aiuti per anno

df_trend = con.execute(f"""
    SELECT anno,
           COUNT(*) AS n_aiuti,
           ROUND(SUM(elemento_aiuto) / 1e9, 2) AS importo_mld
    FROM '{GCS_PATH}'
    WHERE anno >= 2017 AND anno <= 2026
    GROUP BY anno
    ORDER BY anno
""").fetchdf()

print('Trend annuale:')
print(df_trend.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Trend annuale:
 anno  n_aiuti  importo_mld
 2017   216931         4.59
 2018   696534         8.72
 2019   524502         7.50
 2020  2537967        95.80
 2021  2722096       108.07
 2022  2134404        78.81
 2023  2946581        53.16
 2024  2413641        29.69
 2025  2139134        42.92
 2026   643105        51.17


In [3]:
# Figura 1: Trend annuale

fig, ax = plt.subplots(figsize=(12, 5))

colors_bar = ['#95a5a6'] * len(df_trend)
colors_bar[3] = '#e74c3c'   # 2020 - COVID
colors_bar[4] = '#c0392b'   # 2021 - picco

bars = ax.bar(df_trend['anno'].astype(str), df_trend['importo_mld'], color=colors_bar, edgecolor='white', width=0.7)

for bar, val in zip(bars, df_trend['importo_mld']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f mld'))
ax.set_xlabel('Anno')
ax.set_ylabel('Importo totale (miliardi di €)')
ax.set_title('Aiuti di Stato alle imprese italiane — 2017-2026', fontweight='bold', fontsize=15)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../figures/rna_aiuti_stato_trend_annuale.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: rna_aiuti_stato_trend_annuale.png')

Figura salvata: rna_aiuti_stato_trend_annuale.png


In [4]:
# 2. Distribuzione regionale — totale per regione

df_regioni = con.execute(f"""
    SELECT regione_beneficiario AS regione,
           ROUND(SUM(elemento_aiuto) / 1e9, 2) AS importo_mld,
           COUNT(*) AS n_aiuti,
           COUNT(DISTINCT denominazione_beneficiario) AS n_imprese
    FROM '{GCS_PATH}'
    WHERE regione_beneficiario IS NOT NULL AND regione_beneficiario != ''
    GROUP BY regione_beneficiario
    ORDER BY importo_mld DESC
""").fetchdf()

print('Distribuzione regionale (top 10):')
print(df_regioni.head(10).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Distribuzione regionale (top 10):
              regione  importo_mld  n_aiuti  n_imprese
            Lombardia        99.65  2482275     742375
               Veneto        45.59  1418316     372289
                Lazio        42.15  1166745     423383
       Emilia-Romagna        39.63  1165013     353046
             Campania        39.36  1876450     473308
             Piemonte        38.30  1077941     334443
               Puglia        26.77  1210697     276606
              Toscana        26.24  1049561     326611
              Sicilia        25.27  1366548     300405
Friuli-Venezia Giulia        14.34   471516     102380


In [5]:
# Figura 2: Distribuzione regionale

fig, ax = plt.subplots(figsize=(12, 6))

top10 = df_regioni.head(10)
colors_reg = ['#2c3e50'] * 10
colors_reg[0] = '#e74c3c'

bars = ax.barh(top10['regione'][::-1], top10['importo_mld'][::-1], color=colors_reg[::-1], edgecolor='white', height=0.7)

for bar, val in zip(bars, top10['importo_mld'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} mld', ha='left', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Importo totale (miliardi di €)')
ax.set_title('Aiuti di Stato per regione — Top 10 (2017-2026)', fontweight='bold', fontsize=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../figures/rna_aiuti_stato_distribuzione_regionale.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: rna_aiuti_stato_distribuzione_regionale.png')

Figura salvata: rna_aiuti_stato_distribuzione_regionale.png


In [6]:
# 3. Per strumento — come vengono erogati gli aiuti

df_strumenti = con.execute(f"""
    SELECT CASE 
        WHEN strumento LIKE 'Garanzia%' THEN 'Garanzia'
        WHEN strumento LIKE 'Sovvenzione%' OR strumento LIKE 'contributo a fondo%' OR strumento LIKE 'contributo%' THEN 'Sovvenzione/Contributo'
        WHEN strumento LIKE 'Agevolazione fiscale%' THEN 'Agevolazione fiscale'
        WHEN strumento LIKE 'Prestito%' THEN 'Prestito/Anticipo'
        ELSE 'Altro'
    END AS categoria,
           ROUND(SUM(elemento_aiuto) / 1e9, 2) AS importo_mld,
           COUNT(*) AS n_aiuti
    FROM '{GCS_PATH}'
    WHERE strumento IS NOT NULL AND strumento != ''
    GROUP BY 1
    ORDER BY importo_mld DESC
""").fetchdf()

print('Per categoria di strumento:')
print(df_strumenti.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Per categoria di strumento:
             categoria  importo_mld  n_aiuti
              Garanzia       259.20  4493559
Sovvenzione/Contributo       129.62  5127042
  Agevolazione fiscale        53.41  6536685
                 Altro        34.15   639378
     Prestito/Anticipo         4.05   177265


In [7]:
# Figura 3: Strumenti - distribuzione

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_s = ['#2c3e50', '#27ae60', '#e67e22', '#3498db', '#95a5a6']

# Pie chart
wedges, texts, autotexts = ax1.pie(
    df_strumenti['importo_mld'], 
    labels=None,
    autopct='%1.1f%%',
    colors=colors_s[:len(df_strumenti)],
    startangle=90,
    textprops={'fontsize': 11}
)
ax1.set_title('Per valore (miliardi di €)', fontweight='bold')

# Bar chart
bars = ax2.bar(df_strumenti['categoria'], df_strumenti['importo_mld'], color=colors_s[:len(df_strumenti)], edgecolor='white')
for bar, val in zip(bars, df_strumenti['importo_mld']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Miliardi di €')
ax2.set_title('Confronto per categoria', fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.setp(ax2.get_xticklabels(), rotation=20, ha='right')

fig.suptitle('Strumenti di erogazione degli aiuti di Stato', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../figures/rna_aiuti_stato_strumenti.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: rna_aiuti_stato_strumenti.png')

Figura salvata: rna_aiuti_stato_strumenti.png


In [8]:
# 4. Per tipo di beneficiario — PMI vs Grande impresa nel tempo

df_tipo = con.execute(f"""
    SELECT anno,
           tipo_beneficiario,
           ROUND(SUM(elemento_aiuto) / 1e9, 2) AS importo_mld
    FROM '{GCS_PATH}'
    WHERE tipo_beneficiario IN ('PMI', 'Grande impresa')
      AND anno IN (2019, 2020, 2021, 2022, 2023, 2024, 2025)
    GROUP BY anno, tipo_beneficiario
    ORDER BY anno, tipo_beneficiario
""").fetchdf()

print('PMI vs Grande impresa:')
print(df_tipo.to_string(index=False))

PMI vs Grande impresa:
 anno tipo_beneficiario  importo_mld
 2019    Grande impresa         1.31
 2019               PMI         5.92
 2020    Grande impresa         1.31
 2020               PMI        94.09
 2021    Grande impresa        39.24
 2021               PMI        68.52
 2022    Grande impresa        31.48
 2022               PMI        46.38
 2023    Grande impresa        16.87
 2023               PMI        35.83
 2024    Grande impresa         7.10
 2024               PMI        21.63
 2025    Grande impresa        12.61
 2025               PMI        29.96


In [9]:
# Figura 4: PMI vs Grande Impresa — stacked bar

fig, ax = plt.subplots(figsize=(12, 5))

pivot_tipo = df_tipo.pivot_table(index='anno', columns='tipo_beneficiario', values='importo_mld', aggfunc='sum').fillna(0)

ax.bar(pivot_tipo.index.astype(str), pivot_tipo['PMI'], label='PMI', color='#3498db', edgecolor='white')
ax.bar(pivot_tipo.index.astype(str), pivot_tipo['Grande impresa'], bottom=pivot_tipo['PMI'],
       label='Grande impresa', color='#2c3e50', edgecolor='white')

for i, anno in enumerate(pivot_tipo.index):
    totale = pivot_tipo.loc[anno, 'PMI'] + pivot_tipo.loc[anno, 'Grande impresa']
    pct_pmi = pivot_tipo.loc[anno, 'PMI'] / totale * 100
    ax.text(i, totale + 1.5, f'{totale:.1f} mld\nPMI {pct_pmi:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.legend(fontsize=11)
ax.set_ylabel('Miliardi di €')
ax.set_xlabel('Anno')
ax.set_title('Aiuti per tipo di beneficiario: PMI vs Grande Impresa', fontweight='bold', fontsize=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../figures/rna_aiuti_stato_tipo_beneficiario.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: rna_aiuti_stato_tipo_beneficiario.png')

Figura salvata: rna_aiuti_stato_tipo_beneficiario.png


In [10]:
# 5. Per procedimento — De Minimis, Notifica, Esenzione

df_proc = con.execute(f"""
    SELECT procedimento,
           ROUND(SUM(elemento_aiuto) / 1e9, 2) AS importo_mld,
           COUNT(*) AS n_aiuti
    FROM '{GCS_PATH}'
    WHERE procedimento IS NOT NULL AND procedimento != ''
    GROUP BY procedimento
    ORDER BY importo_mld DESC
""").fetchdf()

df_proc['pct_valore'] = df_proc['importo_mld'] / df_proc['importo_mld'].sum() * 100
df_proc['pct_numero'] = df_proc['n_aiuti'] / df_proc['n_aiuti'].sum() * 100

print('Per procedimento:')
print(df_proc.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Per procedimento:
procedimento  importo_mld  n_aiuti  pct_valore  pct_numero
    Notifica       391.74 10654457   81.541151   62.769539
   Esenzione        56.90  1678114   11.843803    9.886421
  De Minimis        31.78  4641358    6.615045   27.344040


In [11]:
# Figura 5: Procedimenti — confronto valore vs numero

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(df_proc))
width = 0.35

bars1 = ax.bar(x - width/2, df_proc['pct_valore'], width, label='% del valore', color='#2c3e50', edgecolor='white')
bars2 = ax.bar(x + width/2, df_proc['pct_numero'], width, label='% del numero di aiuti', color='#e74c3c', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(df_proc['procedimento'])
ax.legend(fontsize=11)
ax.set_ylabel('% sul totale')
ax.set_title('Aiuti per tipo di procedimento: valore vs frequenza', fontweight='bold', fontsize=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../figures/rna_aiuti_stato_procedimenti.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: rna_aiuti_stato_procedimenti.png')

Figura salvata: rna_aiuti_stato_procedimenti.png


In [12]:
# 6. Top concedenti

df_concedenti = con.execute(f"""
    SELECT soggetto_concedente,
           ROUND(SUM(elemento_aiuto) / 1e9, 2) AS importo_mld,
           COUNT(*) AS n_aiuti
    FROM '{GCS_PATH}'
    WHERE soggetto_concedente IS NOT NULL AND soggetto_concedente != ''
    GROUP BY soggetto_concedente
    ORDER BY importo_mld DESC
    LIMIT 10
""").fetchdf()

print('Top 10 concedenti:')
print(df_concedenti.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Top 10 concedenti:
                                                                                                                           soggetto_concedente  importo_mld  n_aiuti
                                                                                            Banca del Mezzogiorno MedioCredito Centrale S.p.A.       199.88  4330654
                                                                                                                                   SACE S.P.A.        58.76    12541
                                                                                                       GSE – Gestore Servizi Energetici S.p.A.        50.15    18009
                                                                                                                         agenzia delle entrate        23.20  3875390
                                                                                                                                          inps        17.85 

In [13]:
# 6. Top concedenti — versione abbreviata per visualizzazione

df_conc_short = df_concedenti.copy()
df_conc_short['nome_corto'] = df_conc_short['soggetto_concedente'].str[:45]

fig, ax = plt.subplots(figsize=(12, 5))

colors_c = ['#e74c3c' if i == 0 else '#2c3e50' if i < 3 else '#95a5a6' for i in range(len(df_conc_short))]
bars = ax.barh(df_conc_short['nome_corto'][::-1], df_conc_short['importo_mld'][::-1], 
              color=colors_c[::-1], edgecolor='white', height=0.7)

for bar, val in zip(bars, df_conc_short['importo_mld'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} mld', ha='left', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Miliardi di €')
ax.set_title('Top 10 soggetti concedenti di aiuti di Stato', fontweight='bold', fontsize=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../figures/rna_aiuti_stato_top_concedenti.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salvata: rna_aiuti_stato_top_concedenti.png')

Figura salvata: rna_aiuti_stato_top_concedenti.png


In [14]:
# Riepilogo finale — numeri chiave

df_tot = con.execute(f"""
    SELECT 
        ROUND(SUM(elemento_aiuto) / 1e9, 2) AS totale_mld,
        COUNT(*) AS totale_aiuti,
        COUNT(DISTINCT denominazione_beneficiario) AS totale_imprese,
        ROUND(AVG(elemento_aiuto), 0) AS importo_medio,
        MIN(anno) AS anno_da,
        MAX(anno) AS anno_a
    FROM '{GCS_PATH}'
    WHERE anno >= 2017
""").fetchdf()

print('RIEPILOGO — RNA Aiuti di Stato 2017-2026')
print(f'Totale erogato: {df_tot["totale_mld"].values[0]:.1f} miliardi di €')
print(f'Numero aiuti: {df_tot["totale_aiuti"].values[0]:,}')
print(f'Imprese beneficiarie: {df_tot["totale_imprese"].values[0]:,}')
print(f'Importo medio per aiuto: {df_tot["importo_medio"].values[0]:.0f} €')
print(f'Periodo: {df_tot["anno_da"].values[0]} - {df_tot["anno_a"].values[0]}')

RIEPILOGO — RNA Aiuti di Stato 2017-2026
Totale erogato: 480.4 miliardi di €
Numero aiuti: 16,974,895
Imprese beneficiarie: 4,300,201
Importo medio per aiuto: 28304 €
Periodo: 2017 - 2026


In [15]:
print('Tutte le figure generate:')
import glob
for f in sorted(glob.glob('../figures/rna_aiuti_stato_*.png')):
    print(f'  ✓ {f}')

Tutte le figure generate:
  ✓ ../figures/rna_aiuti_stato_distribuzione_regionale.png
  ✓ ../figures/rna_aiuti_stato_procedimenti.png
  ✓ ../figures/rna_aiuti_stato_strumenti.png
  ✓ ../figures/rna_aiuti_stato_tipo_beneficiario.png
  ✓ ../figures/rna_aiuti_stato_top_concedenti.png
  ✓ ../figures/rna_aiuti_stato_trend_annuale.png
